In [ ]:
###############################################################################
import sys
import yaml

with open("./Configs/lib.yml", "r") as file:
    config_dict = yaml.safe_load(file)
    for path in config_dict['lib_path']:
        sys.path.append(path)
###############################################################################
import SeisRoutine.catalog as src
import SeisRoutine.waveform as srw
import SeisRoutine.plot as srp
import SeisRoutine.seisbench as srsb
import SeisRoutine.waveform.health_check.spike as spike_checker
import SeisRoutine.config as srconf
import SeisRoutine.statistics as srs

In [ ]:
import seisbench.data as sbd
import seisbench.generate as sbg
import numpy as np
import os
from scipy import signal
from tqdm import tqdm
from scipy.stats import skew
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import seaborn as sns
from obspy.signal.filter import bandpass

# Load Dataset

In [ ]:
timestamp = srconf.timestamp()
context={
    "timestamp": timestamp,
    "np": np,
}
cfg_path = r"./Configs/DataSet-Extra-Parameters-cfg.yml"

cfg = srconf.Config.load(
    cfg_path,
    resolve=True,
)
cfg.resolve(context=context)

dataset = sbd.WaveformDataset(
    path=cfg.dataset.path,
    **cfg.dataset.data_format.to_dict(),
)
phase_dict = srsb.dataset.build_phase_mapper(dataset.metadata.columns)
cfg.resolve(
    context={
        "phase_dict": phase_dict
    }
)

augmentations = srsb.dataset.build_augmentations(
    cfg.quality_statistic.augmentation
)
generator = sbg.GenericGenerator(dataset)
generator.add_augmentations(augmentations)

for augmentation in cfg.quality_statistic.augmentation:
    if augmentation.cls == "seisbench.generate.Filter":
        freqmin, freqmax = augmentation.Wn
    else:
        pass
        # freqmin, freqmax = None, None


metadata = dataset.metadata.copy()
for index, extra in enumerate([
    "Extra_MetaData/PS_selected.pkl",
    "Extra_MetaData/Quality_Parameters.pkl"
]):
    df_manual_picks = pd.read_pickle(
        Path(cfg.dataset.path) / extra
    )
    metadata = pd.merge(
        left=metadata,
        right=df_manual_picks,
        on="trace_name",
        suffixes=('', f'_{index}'),
    )

cols_to_convert = ['Manual_Pick_P', 'Manual_Pick_S']
for col in cols_to_convert:
    metadata[col] = metadata[col].astype("Int32")

dataset._metadata = metadata

# View Data Sample

In [ ]:
n = 268
# n = 257
# n = 256
n = 1
n = 432 # double earthquakes
n = 700 # spike in the start of the S-type wave
n = 846 # double eq
n = 1829 # multiple earthquakes (many)

data = generator[n]['X']
# plt.plot(data.T); plt.show()

# plt.plot(data.T / np.abs(data).max(axis=1) + [2, 0, -2]); plt.show()
# plt.plot(data[1]); plt.show()
plt.plot(data.T[500: 1350]); plt.show()


# keys = [k for k in metadata.keys() if "spike" in k]
# fig, axes = srsb.plot.plot_generator(
#     n=n,
#     generator=generator,
#     target_keys=keys,
# )

In [ ]:
import scipy.stats
for data_1c in data:
    print(
    srw.waveform.SpikeDetector2.detect(
        signal=data_1c,
        kwargs_sliding={
            "window": 3*100,
            "step": 1*100,
            "method": "vectorized",
        },
        kwargs_spike_suspected_skewness={
            "threshold": 3
        },
    ),
    scipy.stats.skew(a=data_1c, bias=False),
    srw.waveform.SpikeDetector2.is_spike_suspected_using_skewness(window=data_1c)
    )

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
import pandas as pd

def df2heatmap(conditions, title=None):
    true_percent = conditions.mean() * 100
    #
    summary = pd.DataFrame({"True %": true_percent,
                            "False %": 100 -true_percent})
    print(summary)
    # ساخت label جدید
    labels = [
        f"{col}\n({true_percent[col]:.1f}%)"
        for col in conditions.columns
    ]
    plt.figure(figsize=(12, 6))
    cmap = ListedColormap(["#beaed4", "#7fc97f"]) # [flase, true]
    cmap = ListedColormap([
        # "#fc8d62",
        # "#66c2a5",
        "#66c2a5",
        "#fc8d62",
    ])
    # cmap='gray_r'
    img = plt.imshow(
        conditions.T,
        aspect='auto',
        cmap=cmap,
        interpolation='nearest'
    )
    plt.gca().set_yticks(
        np.arange(-0.5, conditions.shape[1], 1),
        minor=True,
    )
    plt.grid(which='minor', color='white', linestyle='-', linewidth=2)
    plt.yticks(
        range(len(labels)),
        labels
    )
    plt.xlabel("Sample Index")
    plt.ylabel("Conditions")
    plt.title(title)
    # plt.colorbar(label="Condition")
    cbar = plt.colorbar(img, ticks=[0, 1], label="Condition")
    cbar.ax.set_yticklabels(["False", "True"])

    plt.show()

In [ ]:
file_path_qc = r"d:\DataSets-Local\1405-04-03\Test_Data\Extra_MetaData\Quality_Parameters.pkl"
df_qc = pd.read_pickle(file_path_qc)

In [ ]:
df_qc.columns

In [ ]:
for series_name, series in df_qc.items():
    if ('SNR' in series_name):
        print(series_name)
        array = series.explode().dropna().to_numpy().astype(np.int32)
        try:
            bins = np.arange(array.min()-1, array.max()+1, 1)
            plt.hist(
                array,
                bins=bins,
                edgecolor='k',
            )
            plt.title(series_name)
            plt.show()
        except Exception as error:
            print(error)
        # break
# df_qc

In [ ]:
for series_name, series in df_qc.items():
    if ('spike' in series_name) and (series_name.endswith('index')):
        print(series_name)
        array = series.explode().dropna().to_numpy().astype(np.int32)
        try:
            bins = np.arange(array.min()-1, array.max()+1, 1)
            plt.hist(
                array,
                bins=bins,
                edgecolor='k',
            )
            plt.title(series_name)
            plt.show()
        except Exception as error:
            print(error)
        # break
# df_qc

In [ ]:
file_path = r'c:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisPhaseTune\Training\Configs\Parameters-cfg.yml'
timestamp = srconf.timestamp()
cfg = srconf.Config.load(
    file_path=file_path,
    resolve=True,
)
context={
    "timestamp": timestamp,
}
cfg.resolve(context=context)

In [ ]:
for cha in "ENZ":
    conditions = metadata[[key for key in metadata.keys() if key.startswith(f'trace_{cha}_spike')]]
    metadata[f'all_{cha}'] = conditions.sum(axis=1) >= 4
    conditions = conditions.dropna()
    conditions = conditions.astype(int)
    df2heatmap(conditions, title=None)
# conditions.iloc[1]

In [ ]:
# for idx in range(50, 100):
#     sample = generator[idx]
#     data_3c = sample['X']
#     plt.plot(data_3c[:, :].T + [2, 0, -2], label=['Z', 'N', 'E'])
#     plt.title(str(idx))
#     plt.legend()
#     plt.show()
#     if idx > 100:
#         break

In [ ]:
# [key for key, val in features.items() if ('spike' in key) and (val)]

In [ ]:
# sample = generator[77]
# data_3c = sample['X']
# # plt.plot(data_3c[:, 400:].T + [2, 0, -2], label=['Z', 'N', 'E'])
# # plt.title(str(idx))
# # plt.legend()
# # plt.show()
# c = data_3c[2, :]
# (c.max(),
#  c.min(),
#  spike_checker.spike_by_skewness(c, threshold=3, axis=0),
#  spike_checker.spike_by_kurtosis(c, threshold=100, axis=0, preprocessing=False),
#  min_max_ratio(c),
# )

In [ ]:
# msk = metadata[['all_E', 'all_N', 'all_Z']].sum(axis=1) != 0
# msk = metadata[[f"trace_{cha}_spike: min_max_ration" for cha in "ENZ"] +
#                [f"trace_{cha}_spike: skewness" for cha in "ENZ"]
#                ].sum(axis=1) != 0
# # metadata[msk]

In [ ]:
for idx, features in metadata[msk].iterrows():
    sample = generator[idx]
    data_3c = sample['X']
    label = [key for key, val in features.items() if ('spike' in key) and val]
    label = '\n'.join(label)
    plt.plot(data_3c[:, 400:].T + [2, 0, -2], label=['Z', 'N', 'E'])
    plt.title(str(idx)+label)
    plt.legend()
    plt.show()
    if idx > 100:
        break

In [ ]:
from myfuncs.spike_detection import detect_spikes_ensemble

In [ ]:
conditions[conditions['all']==1].index

In [ ]:
keys = [
    # 'trace_E_spike: zscore',
    # 'trace_E_spike: mad',
    # 'trace_E_spike: wavelet',
    'trace_E_spike: skewness',
    'trace_N_spike: skewness',
    'trace_Z_spike: skewness',
]
msk = metadata[keys].any(axis=1)
for idx, row in metadata[msk].iterrows():
    sample = generator[idx]
    data_3c = sample['X']
    plt.plot(data_3c[:, 400:1000].T + [-2, 0, 2])
    for data_1c in data_3c:
        spike_mask, vote_count = detect_spikes_ensemble(
            data_1c,
            sampling_rate=100.0,
            vote_threshold=3,
        )
        print(spike_mask.any(), vote_count.max())
    plt.title(str(idx))
    plt.show()
    if idx > 100:
        break

# Section 2

In [ ]:
def plot_column_heatbars(df, cmap='jet', figsize=(12, 6), vmin=None, vmax=None):
    """
    Plot each DataFrame column as a horizontal color bar.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame whose columns will be plotted.
    cmap : str
        Matplotlib colormap.
    figsize : tuple
        Figure size.
    vmin, vmax : float or None
        Shared color scale limits. If None, they are computed from the data.
    """

    if vmin is None:
        vmin = np.nanmin(df.values)
    if vmax is None:
        vmax = np.nanmax(df.values)

    fig, ax = plt.subplots(figsize=figsize)

    ax.imshow(
        df.T,
        aspect='auto',
        cmap=cmap,
        interpolation='nearest',
        norm="log"
        # vmin=vmin,
        # vmax=vmax
    )

    ax.set_yticks(np.arange(len(df.columns)))
    ax.set_yticklabels(df.columns)

    ax.set_xlabel("Sample index")
    ax.set_ylabel("Feature")

    cbar = plt.colorbar(ax.images[0], ax=ax)
    cbar.set_label("Standardized value")
    plt.gca().set_yticks(
        np.arange(-0.5, df.shape[1], 1),
        minor=True
    )
    plt.grid(which='minor', color='white', linestyle='-', linewidth=2)

    plt.tight_layout()
    plt.show()

In [ ]:
path = Path(r"D:\DataSets-Local\1405-04-03\Merged_Dataset_2026-06-24T15-15-22\evaluations")

df = pd.read_csv(path / "All_Metrics_Final.csv")

keys = [col for col in df.columns if col.startswith('SNR_Z')]
df_snr_z = df[keys]

# df_plot = df_snr_z/df_snr_z.abs().max()
# df_plot = df_snr_z.sub(df_snr_z.mean(axis=1), axis=0)
# df_plot = df_snr_z.sub(df_snr_z.mean(axis=1), axis=0).div(df_snr_z.std(axis=1), axis=0)
df_plot = df_snr_z.rank(axis=1)
df_plot = df_snr_z.sub(
    df_snr_z.mean(axis=1), axis=0
    ).div(
        df_snr_z.std(axis=1), axis=0
)

plot_column_heatbars(df_plot,
                     cmap='jet', figsize=(12, 6), vmin=None, vmax=None)


df_plot = df_snr_z.corr()
sns.heatmap(df_plot.head(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
proj = pca.fit_transform(df_snr_z.T)

plt.scatter(proj[:,0], proj[:,1])


mean = (df_snr_z['SNR_Z_power_in_time'] + df_snr_z['SNR_Z_mad']) / 2
diff = df_snr_z['SNR_Z_power_in_time'] - df_snr_z['SNR_Z_mad']
x = mean
y = diff
plt.plot(x, y)